<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-06-rag-systems/lesson-6.11-rag-evaluation-and-ragas/notebooks/exercises-6_11.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 6.11: RAG Evaluation with RAGAS

*Module 6 · Retrieval-Augmented Generation (RAG)*

> You've built 10 versions of RAG. Which one is actually best? Without rigorous evaluation, you're guessing. RAGAS gives you 4 metrics that tell you exactly where your RAG system is strong, weak, and hallucinating.

---

## About this notebook

This notebook contains the **4 runnable code examples** from the published lesson `lesson-6.11.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — The 4 Metrics From Scratch — Understand Before You Automate — `part1_metrics_scratch.py`
2. Step 2 — RAGAS Library — Automated Evaluation at Scale — `part2_ragas_library.py`
3. Step 3 — Evaluation Dataset — Build Your Test Set — `part3_eval_dataset.py`
4. Step 4 — Continuous Improvement Loop — A/B Test Every Change — `part4_improvement_loop.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai transformers langchain ragas datasets


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · The 4 Metrics From Scratch — Understand Before You Automate

Build each RAGAS metric manually so you know exactly what the framework measures.

**`part1_metrics_scratch.py`** · _Block 1 of 4_

4 RAGAS METRICS — BUILT FROM SCRATCH — Understand how each metric works before


In [ ]:
# =============================================
# 4 RAGAS METRICS — BUILT FROM SCRATCH
# Understand how each metric works before
# using the framework.
# =============================================

import google.generativeai as genai
import json, re, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class RAGEvaluator:
    """Evaluate RAG quality with 4 core metrics (RAGAS-style)."""
    
    def __init__(self):
        self.judge = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0.0})
    
    # ── Metric 1: FAITHFULNESS ──
    # "Is every claim in the answer supported by the context?"
    # Catches hallucination. High = grounded. Low = making things up.
    
    def faithfulness(self, answer: str, contexts: list[str]) -> float:
        """Score 0-1: Is the answer supported by the retrieved context?"""
        context_text = "\n\n".join([f"[Context {i+1}] {c}" for i, c in enumerate(contexts)])
        
        prompt = f"""Break the answer into individual claims, then check each against the context.

Answer: "{answer}"

Context:
{context_text}

Return ONLY valid JSON:
{{
  "claims": [
    {{"claim": "text of claim", "supported": true/false, "evidence": "which context supports it or 'none'"}}
  ],
  "faithfulness_score": 0.0-1.0
}}

Score = (supported claims) / (total claims). If all claims are supported, score = 1.0."""
        
        resp = self.judge.generate_content(prompt)
        try:
            data = json.loads(re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip())
            return float(data.get("faithfulness_score", 0.5))
        except:
            return 0.5
    
    # ── Metric 2: ANSWER RELEVANCE ──
    # "Does the answer actually address the question?"
    # Catches off-topic or partial answers.
    
    def answer_relevance(self, question: str, answer: str) -> float:
        """Score 0-1: Does the answer address the question asked?"""
        prompt = f"""Rate how well this answer addresses the question.

Question: "{question}"
Answer: "{answer}"

Return ONLY valid JSON:
{{
  "relevance_score": 0.0-1.0,
  "reasoning": "brief explanation"
}}

Scoring:
  1.0 = directly and completely answers the question
  0.7 = partially answers, missing some aspects
  0.3 = tangentially related but doesn't answer
  0.0 = completely irrelevant"""
        
        resp = self.judge.generate_content(prompt)
        try:
            data = json.loads(re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip())
            return float(data.get("relevance_score", 0.5))
        except:
            return 0.5
    
    # ── Metric 3: CONTEXT PRECISION ──
    # "Are the retrieved chunks actually relevant to the question?"
    # Measures retrieval quality. High = right chunks found.
    
    def context_precision(self, question: str, contexts: list[str]) -> float:
        """Score 0-1: What fraction of retrieved contexts are relevant?"""
        prompt = f"""For each retrieved context, judge if it's relevant to answering the question.

Question: "{question}"

Contexts:
{chr(10).join(f"[{i+1}] {c[:200]}" for i, c in enumerate(contexts))}

Return ONLY valid JSON:
{{
  "judgments": [
    {{"context_index": 1, "relevant": true/false, "reason": "brief"}}
  ],
  "precision_score": 0.0-1.0
}}

Score = (relevant contexts) / (total contexts retrieved)."""
        
        resp = self.judge.generate_content(prompt)
        try:
            data = json.loads(re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip())
            return float(data.get("precision_score", 0.5))
        except:
            return 0.5
    
    # ── Metric 4: CONTEXT RECALL ──
    # "Did we find ALL the relevant information?"
    # Needs ground-truth answer. Measures completeness.
    
    def context_recall(self, ground_truth: str, contexts: list[str]) -> float:
        """Score 0-1: What fraction of the ground-truth is covered by contexts?"""
        prompt = f"""Break the ground-truth answer into key facts. Check if each fact appears in the contexts.

Ground-truth answer: "{ground_truth}"

Retrieved contexts:
{chr(10).join(f"[{i+1}] {c[:200]}" for i, c in enumerate(contexts))}

Return ONLY valid JSON:
{{
  "facts": [
    {{"fact": "key fact from ground truth", "found_in_context": true/false}}
  ],
  "recall_score": 0.0-1.0
}}

Score = (facts found in contexts) / (total facts in ground truth)."""
        
        resp = self.judge.generate_content(prompt)
        try:
            data = json.loads(re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip())
            return float(data.get("recall_score", 0.5))
        except:
            return 0.5
    
    # ── Run all 4 metrics ──
    def evaluate(self, question: str, answer: str, contexts: list[str],
                 ground_truth: str = "") -> dict:
        """Run all RAGAS metrics on a single Q&A pair."""
        scores = {
            "faithfulness": self.faithfulness(answer, contexts),
            "answer_relevance": self.answer_relevance(question, answer),
            "context_precision": self.context_precision(question, contexts),
        }
        if ground_truth:
            scores["context_recall"] = self.context_recall(ground_truth, contexts)
        
        scores["overall"] = sum(scores.values()) / len(scores)
        return scores

# ── Demo: Evaluate a RAG response ──
evaluator = RAGEvaluator()

result = evaluator.evaluate(
    question="What is the refund policy?",
    answer="Refunds are processed within 14 business days. You need to submit form RF-201 with your order ID. The company was founded in 2020.",
    contexts=[
        "Refunds are processed within 14 business days. Customers must submit form RF-201.",
        "The platform supports Python 3.9+ and requires Docker.",
    ],
    ground_truth="Refunds take 14 business days. Form RF-201 is required with the order ID.",
)

print("RAG Evaluation Results:\n")
for metric, score in result.items():
    bar = "█" * int(score * 20) + "░" * (20 - int(score * 20))
    print(f"  {metric:22s} {bar} {score:.2f}")

print("""
What each score reveals about this response:
  faithfulness ~0.67    → "founded in 2020" is NOT in the context = hallucination!
  answer_relevance ~0.9 → Answer directly addresses the refund question
  context_precision ~0.5→ Only 1 of 2 retrieved contexts was relevant (Python/Docker was noise)
  context_recall ~1.0   → All ground-truth facts were found in the contexts
""")


> **What just happened?** We built all 4 RAGAS metrics from scratch: (1) Faithfulness — breaks the answer into claims, checks each against context, catches "founded in 2020" as hallucination. (2) Answer Relevance — does the answer address the question? (3) Context Precision — were the retrieved chunks relevant? The Python/Docker chunk was noise. (4) Context Recall — did we find all the information from the ground truth? Each metric is a separate LLM judge call with structured JSON output. Now you know exactly what RAGAS measures before using the library.


### Step 2 · RAGAS Library — Automated Evaluation at Scale

Same metrics, but the library handles batching, LLM calls, and reporting.

**`part2_ragas_library.py`** · _Block 2 of 4_

RAGAS LIBRARY: Automated evaluation — pip install ragas datasets langchain-google-genai


In [ ]:
# =============================================
# RAGAS LIBRARY: Automated evaluation
# pip install ragas datasets langchain-google-genai
# =============================================

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from datasets import Dataset

# ── Configure RAGAS to use Gemini ──
llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(model="gemini-2.0-flash"))
embeddings = LangchainEmbeddingsWrapper(GoogleGenerativeAIEmbeddings(model="models/text-embedding-004"))

# ── Build the evaluation dataset ──
eval_data = {
    "question": [
        "What is the refund policy?",
        "What GPU is required?",
        "How much does the course cost?",
        "What modules does the course cover?",
    ],
    "answer": [
        "Refunds are processed within 14 business days via form RF-201.",
        "You need CUDA 12.1+ and 8 GB VRAM minimum for GPU acceleration.",
        "The course costs ₹29,999 annually or ₹2,999 per month.",
        "The course covers transformers, RAG, fine-tuning, agents, and deployment.",
    ],
    "contexts": [
        ["Refunds processed within 14 business days. Submit form RF-201 with order ID."],
        ["GPU acceleration requires CUDA 12.1+. Minimum 8 GB VRAM for inference."],
        ["NetsAI costs ₹29,999/year or ₹2,999/month.", "Enterprise starts at ₹4,99,000/year."],
        ["Course covers 14 modules: transformers, RAG, fine-tuning, agents, GCP."],
    ],
    "ground_truth": [
        "Refunds take 14 business days. Form RF-201 required with order ID.",
        "CUDA 12.1 or later. 8 GB VRAM minimum.",
        "₹29,999 annual plan. ₹2,999 monthly plan.",
        "14 modules: transformers, RAG, fine-tuning, agents, deployment on GCP.",
    ],
}

dataset = Dataset.from_dict(eval_data)

# ── Run evaluation ──
print("Running RAGAS evaluation...\n")
results = evaluate(
    dataset=dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=llm,
    embeddings=embeddings,
)

print("RAGAS Results:")
print(results)

# Per-question breakdown
df = results.to_pandas()
print("\nPer-question scores:")
print(df[["question", "faithfulness", "answer_relevancy", "context_precision", "context_recall"]].to_string(index=False))


> **What just happened?** We ran the official RAGAS library with Gemini as the judge. Four questions, four answers, four sets of contexts, four ground truths — RAGAS computed all 4 metrics for each and returned a DataFrame with per-question scores. This is what production RAG teams run after every pipeline change: swap a chunking strategy → re-run RAGAS → compare scores. If faithfulness drops, the new strategy caused more hallucination.


### Step 3 · Evaluation Dataset — Build Your Test Set

Good evaluation needs good test data. Generate it semi-automatically from your documents.

**`part3_eval_dataset.py`** · _Block 3 of 4_

EVALUATION DATASET GENERATOR — Semi-automatically create QA pairs from docs.


In [ ]:
# =============================================
# EVALUATION DATASET GENERATOR
# Semi-automatically create QA pairs from docs.
# =============================================

class EvalDatasetGenerator:
    """Generate evaluation QA pairs from document chunks."""
    
    def __init__(self):
        self.model = genai.GenerativeModel("gemini-2.0-flash",
            generation_config={"temperature": 0.4})
    
    def generate_qa_pairs(self, chunk: str, n_questions: int = 3) -> list[dict]:
        """Generate question-answer pairs from a chunk of text."""
        prompt = f"""Generate {n_questions} diverse question-answer pairs from this text.

Text: "{chunk}"

Return ONLY valid JSON:
{{
  "pairs": [
    {{
      "question": "a natural question someone would ask",
      "answer": "the correct answer based ONLY on the text",
      "difficulty": "easy|medium|hard"
    }}
  ]
}}

Rules:
- Questions should be natural (how a real user would ask)
- Include a mix: factual, comparison, and reasoning questions
- Answers must be FULLY supported by the text
- Include at least one "hard" question that requires inference"""
        
        resp = self.model.generate_content(prompt)
        try:
            data = json.loads(re.sub(r"```json?\s*", "", resp.text).replace("```", "").strip())
            pairs = data.get("pairs", [])
            for p in pairs:
                p["source_chunk"] = chunk
            return pairs
        except:
            return []
    
    def build_dataset(self, chunks: list[str], questions_per_chunk: int = 2) -> list[dict]:
        """Build a full evaluation dataset from multiple chunks."""
        dataset = []
        for i, chunk in enumerate(chunks):
            print(f"  Generating from chunk {i+1}/{len(chunks)}...")
            pairs = self.generate_qa_pairs(chunk, n_questions=questions_per_chunk)
            dataset.extend(pairs)
        
        print(f"\n  ✅ Generated {len(dataset)} QA pairs from {len(chunks)} chunks")
        return dataset

# ── Generate eval dataset from our corpus ──
gen = EvalDatasetGenerator()

sample_chunks = [
    "The NetsAI GenAI course costs ₹29,999 for the annual plan and ₹2,999/month. Enterprise pricing starts at ₹4,99,000/year for up to 50 seats.",
    "Refunds are processed within 14 business days. Customers must submit form RF-201 with their order ID. Student discounts of 20% are available with a valid .edu email.",
    "GPU acceleration requires CUDA 12.1 or later. The minimum VRAM requirement is 8 GB for inference. The free trial lasts 14 days with access to all features except GPU.",
]

eval_dataset = gen.build_dataset(sample_chunks, questions_per_chunk=3)

print("\nGenerated QA pairs:")
for qa in eval_dataset[:4]:
    print(f"  [{qa['difficulty']:6s}] Q: {qa['question']}")
    print(f"           A: {qa['answer'][:80]}...\n")


> **What just happened?** We built an EvalDatasetGenerator that uses an LLM to generate question-answer pairs from document chunks. From 3 chunks, it produced ~9 diverse QA pairs with easy/medium/hard difficulty labels. Each pair includes the source chunk (ground truth context). This saves weeks of manual labeling — you can generate 100+ test questions in minutes, then have a human review and filter them.


### Step 4 · Continuous Improvement Loop — A/B Test Every Change

Change the chunking? Swap the model? New prompt? Measure the impact with RAGAS before deploying.

**`part4_improvement_loop.py`** · _Block 4 of 4_

CONTINUOUS IMPROVEMENT LOOP — A/B test RAG changes with automated metrics.


In [ ]:
# =============================================
# CONTINUOUS IMPROVEMENT LOOP
# A/B test RAG changes with automated metrics.
# =============================================

class RAGExperiment:
    """Run and compare RAG experiments with RAGAS metrics."""
    
    def __init__(self, eval_dataset: list[dict]):
        self.eval_data = eval_dataset
        self.evaluator = RAGEvaluator()
        self.results = {}
    
    def run_experiment(self, name: str, rag_fn) -> dict:
        """Run a RAG pipeline on the eval dataset and score it."""
        print(f"\n🧪 Experiment: {name}")
        
        all_scores = []
        for i, qa in enumerate(self.eval_data):
            # Run the RAG pipeline
            result = rag_fn(qa["question"])
            answer = result.get("answer", "")
            contexts = result.get("contexts", [])
            
            # Evaluate
            scores = self.evaluator.evaluate(
                question=qa["question"],
                answer=answer,
                contexts=contexts,
                ground_truth=qa.get("answer", ""),
            )
            all_scores.append(scores)
        
        # Aggregate scores
        metrics = {}
        for key in all_scores[0]:
            values = [s[key] for s in all_scores if key in s]
            metrics[key] = sum(values) / len(values)
        
        self.results[name] = metrics
        return metrics
    
    def compare(self):
        """Compare all experiment results side by side."""
        if len(self.results) < 2:
            print("Need at least 2 experiments to compare.")
            return
        
        print(f"\n📊 Experiment Comparison ({len(self.eval_data)} test questions)")
        print("═" * 75)
        
        metrics = ["faithfulness", "answer_relevance", "context_precision", "context_recall", "overall"]
        header = f"{'Metric':<22}" + "".join([f"{name:>14}" for name in self.results])
        print(header)
        print("─" * 75)
        
        for metric in metrics:
            row = f"  {metric:<22}"
            values = []
            for name in self.results:
                val = self.results[name].get(metric, 0)
                values.append(val)
                row += f"{val:>13.1%}"
            
            # Mark improvements
            if len(values) >= 2:
                diff = values[-1] - values[0]
                icon = "📈" if diff > 0.02 else ("📉" if diff < -0.02 else "➡️")
                row += f"  {icon} {diff:+.1%}"
            
            print(row)
        
        # Recommendation
        names = list(self.results.keys())
        best = max(names, key=lambda n: self.results[n].get("overall", 0))
        print(f"\n  ✅ Winner: {best} (highest overall score)")
    
    def diagnose(self, name: str):
        """Diagnose weaknesses in a specific experiment."""
        scores = self.results.get(name, {})
        if not scores: return
        
        print(f"\n🔍 Diagnosis for '{name}':")
        
        if scores.get("faithfulness", 1) < 0.7:
            print("  ⚠️ LOW FAITHFULNESS — the model is hallucinating.")
            print("     Fix: stronger grounding prompt, lower temperature, add 'only from context' instruction")
        
        if scores.get("context_precision", 1) < 0.7:
            print("  ⚠️ LOW CONTEXT PRECISION — retriever is finding irrelevant chunks.")
            print("     Fix: better chunking, try hybrid search, add re-ranking")
        
        if scores.get("context_recall", 1) < 0.7:
            print("  ⚠️ LOW CONTEXT RECALL — retriever is missing relevant chunks.")
            print("     Fix: increase top_k, try multi-query retrieval, smaller chunks")
        
        if scores.get("answer_relevance", 1) < 0.7:
            print("  ⚠️ LOW ANSWER RELEVANCE — answers are off-topic.")
            print("     Fix: improve prompt template, add 'answer the question directly' instruction")
        
        if all(v >= 0.7 for v in scores.values()):
            print("  ✅ All metrics healthy! Focus on edge cases and harder test questions.")

# ═══════════════════════════════════════════
# EXAMPLE: Compare two RAG configurations
# ═══════════════════════════════════════════

# Build a small eval set
test_set = [
    {"question": "What is the refund policy?", "answer": "14 business days, form RF-201 required."},
    {"question": "What GPU is required?", "answer": "CUDA 12.1+, 8 GB VRAM minimum."},
    {"question": "How much does it cost?", "answer": "₹29,999/year or ₹2,999/month."},
]

experiment = RAGExperiment(test_set)

# Mock RAG pipelines (replace with your actual pipelines)
def rag_v1(question):
    return {"answer": "Refunds take 14 days. Form RF-201 needed.",
            "contexts": ["Refunds processed within 14 business days."]}

def rag_v2(question):
    return {"answer": "Refunds take 14 business days. Submit form RF-201 with your order ID.",
            "contexts": ["Refunds processed within 14 business days. Submit RF-201 with order ID."]}

experiment.run_experiment("RAG v1 (basic)", rag_v1)
experiment.run_experiment("RAG v2 (improved)", rag_v2)
experiment.compare()
experiment.diagnose("RAG v1 (basic)")


> **What just happened?** We built a complete RAGExperiment system: (1) Run two RAG pipelines on the same test questions. (2) Compute all 4 RAGAS metrics for each. (3) Compare side-by-side with improvement/regression indicators (📈/📉). (4) Auto-diagnose weaknesses: "LOW FAITHFULNESS → hallucinating, fix: stronger grounding prompt." This is the production workflow: measure baseline → make a change → re-evaluate → deploy if better. Every RAG improvement is backed by data.


---

## Wrap-up

You've walked through all 4 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
